# 🔒 IEEE-CIS Fraud Detection — Advanced Data Preprocessing Pipeline

> **Goal:** Load the raw transaction and identity CSVs, apply a rigorous object-oriented
> preprocessing pipeline, and export two ready-to-use datasets:
> - `data/processed_train.csv` — full unbalanced data → **use for EDA**
> - `data/balanced_train.csv`  — SMOTE-balanced data  → **use for model training**

### Pipeline Overview

| Step | Transformer | Purpose |
|------|------------|--------|
| 1 | `reduce_mem_usage` | Downcast dtypes — shrink RAM by ~50 % |
| 2 | `DataMerger` | Left-join Transaction + Identity; fix `id-XX` → `id_XX` |
| 3 | `TimeFeatureExtractor` | Extract hour-of-day & day-of-week from `TransactionDT` |
| 4 | `DropHighNulls` | Remove columns with > 85 % missing values |
| 5 | `FrequencyEncoder` | Replace categories with their relative frequency |
| 6 | Final cast + NaN fill | Ensure all columns are numeric (required by SMOTE) |
| 7 | Save EDA CSV | Write `data/processed_train.csv` (unbalanced) |
| 8 | `ClassImbalanceHandler` | SMOTE oversampling + random undersampling |
| 9 | Save balanced CSV | Write `data/balanced_train.csv` (for modelling) |

---
Each transformer is built as a **scikit-learn custom estimator** (`BaseEstimator` + `TransformerMixin`),
so the full pipeline is reusable, stateful, and leak-free when applied to the test set.

## ⚙️ Step 0 — Imports & Path Configuration

We import the standard scientific Python stack plus scikit-learn's base classes that allow us
to write custom, pipeline-compatible transformers.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR     = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR     = os.path.join(BASE_DIR, 'data')
EDA_CSV      = os.path.join(DATA_DIR, 'processed_train.csv')
BALANCED_CSV = os.path.join(DATA_DIR, 'balanced_train.csv')

print('Project root :', BASE_DIR)
print('Data dir     :', DATA_DIR)

## 🗜️ Step 1 — Memory Optimisation (`reduce_mem_usage`)

### What it does
Iterates every numeric column and downcasts it to the smallest dtype that can safely hold its
value range (e.g. `float64` → `float16`, `int64` → `int8`).

### Why it is necessary
The raw CSVs total **~1.5 GB**. Pandas defaults to 64-bit types for everything, often wasting
4× the RAM actually required. Downcasting typically achieves a **50–70 % memory reduction**,
making local experimentation feasible on a standard laptop without kernel crashes.

In [ ]:
def reduce_mem_usage(df, verbose=True):
    """Automatically downcast numeric columns to the minimum safe dtype."""
    numerics  = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type.name in numerics:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                for t in [np.int8, np.int16, np.int32, np.int64]:
                    if np.iinfo(t).min < c_min and c_max < np.iinfo(t).max:
                        df[col] = df[col].astype(t); break
            else:
                for t in [np.float16, np.float32, np.float64]:
                    if np.finfo(t).min < c_min and c_max < np.finfo(t).max:
                        df[col] = df[col].astype(t); break
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f'  RAM: {start_mem:.1f} MB  →  {end_mem:.1f} MB  '
              f'({100*(start_mem-end_mem)/start_mem:.1f}% reduction)')
    return df

## 🔀 Step 2 — Data Merger (`DataMerger`)

### What it does
Performs a **left join** of the Transaction table onto the Identity table using `TransactionID`
as the join key. Also standardises identity column names (`id-01` → `id_01`).

### Why it is necessary
Fraud footprints rely heavily on device and network metadata stored only in the identity table.
Not every transaction has associated identity data — a left join preserves all transactions.
The dash/underscore mismatch between train and test identity columns would silently break
model inference at scoring time if left unfixed.

In [ ]:
class DataMerger(BaseEstimator, TransformerMixin):
    """Left-join Transaction + Identity on 'TransactionID'; standardise column names."""
    def __init__(self, identity_df):
        self.identity_df = identity_df

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        print('  [DataMerger] Standardising names and merging ...')
        id_df = self.identity_df.copy()
        id_df.columns = [c.replace('-', '_') for c in id_df.columns]
        merged = X.merge(id_df, on='TransactionID', how='left')
        print(f'  Shape after merge: {merged.shape}')
        return merged

## ⏰ Step 3 — Temporal Feature Engineering (`TimeFeatureExtractor`)

### What it does
Derives **`Transaction_Hour`** (0–23) and **`Transaction_Day`** (0–6) from the raw
`TransactionDT` column (timedelta in seconds from an unknown reference datetime).

### Why it is necessary
Raw seconds form a monotonically increasing counter. Deriving hour-of-day and day-of-week
exposes temporal fraud signatures — e.g. card-testing spikes at 3 AM — with near-zero
computational cost.

In [ ]:
class TimeFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extract hour-of-day and day-of-week from TransactionDT."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        print('  [TimeFeatureExtractor] Engineering temporal features ...')
        X = X.copy()
        X['Transaction_Hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_Day']  = (X['TransactionDT'] // 86400) % 7
        return X

## 🗑️ Step 4 — High-Null Column Pruning (`DropHighNulls`)

### What it does
During **`fit()`**, records columns exceeding the null threshold.
During **`transform()`**, drops those columns from any DataFrame passed in.

### Why it is necessary
Many `V`-series features have > 90 % missing values. Imputing them forces the model to
learn an imputed constant instead of a real signal, injecting noise and worsening the
*curse of dimensionality*.

> **Threshold:** `0.85` — drop any column missing more than 85 % of its values.

In [ ]:
class DropHighNulls(BaseEstimator, TransformerMixin):
    """Drop columns whose null fraction exceeds `threshold`."""
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        null_frac = X.isnull().mean()
        self.cols_to_drop_ = null_frac[null_frac > self.threshold].index.tolist()
        print(f'  [DropHighNulls] {len(self.cols_to_drop_)} columns flagged '
              f'(>{self.threshold*100:.0f}% missing)')
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors='ignore')

## 🔡 Step 5 — Frequency Encoding (`FrequencyEncoder`)

### What it does
Replaces each category with its **normalised frequency** as learned from the training set.

### Why it is necessary
| Encoding | Problem |
|----------|---------|
| Label Encoding | Arbitrary integers mislead ordinal-sensitive algorithms |
| One-Hot Encoding | Hundreds of sparse binary columns — memory explosion |
| **Frequency Encoding** | One meaningful numeric per category — rare fraud fingerprints get small, distinctive values |

Fitted on training set only → **no data leakage** to the test set.

In [ ]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Replace categories with their normalized frequency (fitted on training data)."""
    def __init__(self, cat_cols=None):
        self.cat_cols   = cat_cols
        self.freq_maps_ = {}

    def fit(self, X, y=None):
        cols = self.cat_cols or X.select_dtypes(include=['object','category']).columns.tolist()
        for col in cols:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(normalize=True, dropna=False).to_dict()
        print(f'  [FrequencyEncoder] Fitted on {len(self.freq_maps_)} columns')
        return self

    def transform(self, X):
        print(f'  [FrequencyEncoder] Encoding {len(self.freq_maps_)} columns ...')
        X = X.copy()
        for col, fmap in self.freq_maps_.items():
            if col in X.columns:
                X[col] = X[col].map(fmap).fillna(0)
        return X

## ⚖️ Step 6 — Class Imbalance Handler (`ClassImbalanceHandler`)

### What it does
Applies a **two-phase** resampling strategy to produce a balanced training set:

**Phase 1 — SMOTE (Synthetic Minority Over-sampling Technique)**
Generates *synthetic* fraud samples by interpolating between existing minority-class
neighbours in feature space. Superior to duplicating rows — adds diversity, avoids
overfitting to exact existing fraud patterns.

**Phase 2 — Random Undersampling**
Randomly removes majority-class (legit) rows until a target 1:1 ratio is achieved,
further balancing the set and reducing training time.

### Why it is necessary
The dataset has a **~97:3 imbalance** (legit:fraud). Without correction:
- A model always predicting 'legit' achieves **97 % accuracy** but catches **0 % of fraud**.
- Standard losses update mostly on majority-class examples, making the model blind to fraud signals.

> ⚠️ **Critical rule:** Apply `ClassImbalanceHandler` **ONLY to training data** — never to
> validation or test sets. SMOTE generates synthetic points not drawn from the real distribution;
> using them in evaluation would corrupt your metrics.

### Resampling strategy
| Phase | Parameter | Effect |
|-------|-----------|--------|
| SMOTE | `sampling_strategy=0.2` | Brings fraud to 20 % of majority count |
| Undersampling | `sampling_strategy=0.5` | Final set is 50 % legit / 50 % fraud |

In [ ]:
from imblearn.over_sampling  import SMOTE
from imblearn.under_sampling import RandomUnderSampler

class ClassImbalanceHandler:
    """
    Two-phase class balancer: SMOTE oversampling → Random undersampling.
    Must ONLY be called on training data — never on validation or test sets.
    """
    def __init__(self, smote_ratio=0.2, under_ratio=0.5, random_state=42, use_undersample=True):
        self.smote_ratio     = smote_ratio
        self.under_ratio     = under_ratio
        self.random_state    = random_state
        self.use_undersample = use_undersample

    def fit_resample(self, X, y):
        cols = X.columns
        print(f'  Before             : {y.value_counts().to_dict()}')

        # Phase 1 — SMOTE (n_jobs removed for compatibility with newer imblearn versions)
        smote = SMOTE(sampling_strategy=self.smote_ratio,
                      random_state=self.random_state)
        X_np, y_np = smote.fit_resample(X.values, y.values)
        print(f'  After SMOTE        : {pd.Series(y_np).value_counts().to_dict()}')

        # Phase 2 — Random undersampling
        if self.use_undersample:
            rus = RandomUnderSampler(sampling_strategy=self.under_ratio,
                                     random_state=self.random_state)
            X_np, y_np = rus.fit_resample(X_np, y_np)
            print(f'  After undersampling: {pd.Series(y_np).value_counts().to_dict()}')

        X_res = pd.DataFrame(X_np, columns=cols)
        y_res = pd.Series(y_np, name=y.name)
        print(f'  Final ratio        : {(y_res==0).sum():,} legit | {(y_res==1).sum():,} fraud')
        return X_res, y_res

## 🚀 Step 7 — Load Data & Run the Complete Pipeline

Now that all transformers and handlers are defined:
1. **Read** both raw CSVs.
2. **Memory-optimise** them immediately.
3. **Run** the scikit-learn `Pipeline`.
4. **Cast** residual object columns → numeric and fill NaNs (SMOTE requires clean numerics).
5. **Save** `processed_train.csv` for EDA.
6. **Balance** via `ClassImbalanceHandler`.
7. **Save** `balanced_train.csv` for model training.

In [ ]:
print('[1/6] Loading raw CSVs ...')
train_tx = pd.read_csv(os.path.join(DATA_DIR, 'train_transaction.csv'))
train_id = pd.read_csv(os.path.join(DATA_DIR, 'train_identity.csv'))
print(f'  train_transaction : {train_tx.shape}')
print(f'  train_identity    : {train_id.shape}')

In [ ]:
print('[2/6] Optimising memory ...')
train_tx = reduce_mem_usage(train_tx)
train_id = reduce_mem_usage(train_id)

In [ ]:
print('[3/6] Running preprocessing pipeline ...')
pipeline = Pipeline([
    ('merger',       DataMerger(identity_df=train_id)),
    ('time_extract', TimeFeatureExtractor()),
    ('drop_nulls',   DropHighNulls(threshold=0.85)),
    ('freq_encoder', FrequencyEncoder()),
])
processed = pipeline.fit_transform(train_tx)

In [ ]:
print('[4/6] Final numeric cast + NaN fill ...')
for col in processed.select_dtypes(include=['object']).columns:
    processed[col] = pd.to_numeric(processed[col], errors='coerce')
# Fill residual NaNs with column median — SMOTE cannot handle NaN values
processed.fillna(processed.median(numeric_only=True), inplace=True)

print(f'  Processed shape  : {processed.shape}')
print('  isFraud BEFORE balancing:')
print(processed['isFraud'].value_counts())
processed.head(3)

In [ ]:
print(f'[5/6] Saving EDA dataset → {EDA_CSV}')
processed.to_csv(EDA_CSV, index=False)
print('  ✅  processed_train.csv saved  (unbalanced — for EDA only)')

In [ ]:
print('[6/6] Handling class imbalance (SMOTE + undersampling) ...')
y = processed['isFraud'].astype(int)
X = processed.drop(columns=['isFraud', 'TransactionID'], errors='ignore')

handler = ClassImbalanceHandler(
    smote_ratio=0.2,      # raise fraud to 20 % of majority via SMOTE
    under_ratio=0.5,      # then undersample majority to reach 1:1
    random_state=42,
    use_undersample=True,
)
X_bal, y_bal = handler.fit_resample(X, y)

balanced = X_bal.copy()
balanced['isFraud'] = y_bal.values

print(f'\nBalanced shape  : {balanced.shape}')
print('isFraud AFTER balancing:')
print(balanced['isFraud'].value_counts())

balanced.to_csv(BALANCED_CSV, index=False)
print(f'\n✅  balanced_train.csv saved  (use this for model training)')
print('\n── Summary ──────────────────────────────────────────')
print('  processed_train.csv  — unbalanced  (EDA / analysis)')
print('  balanced_train.csv   — 1:1 balanced (model training)')